# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and `@id`s.

In [ ]:
print("Record Sets in the dataset:")
record_sets = metadata.get_by_type('RecordSet')
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} : {rs.get('name','(no name)')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', None)
        else:
            field_id = field
        print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs['@id'] for rs in metadata.get_by_type('RecordSet')]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"Record set {record_set_id} returned no records.")
# Choose the first record set for demonstration
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"Columns in main record set ({main_record_set_id}): {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. Operations like removing outliers, transforming columns, or aggregating/grouping are demonstrated.

In [ ]:
# Pick a numeric field for analysis, for demonstration
import numpy as np

numeric_fields = []
# Heuristically guess numeric fields by dtype or by typical protein table field naming conventions
if main_record_set_id:
    df = dataframes[main_record_set_id]
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or (
        col.lower().startswith(('intensity', 'abundance', 'mw', 'count', 'coverage', 'peptide')))]
    print(f"Numeric candidate fields: {numeric_fields}")
    # Use the first numeric field
    numeric_field = numeric_fields[0] if len(numeric_fields) > 0 else df.columns[0]

    # Remove NaNs for calculation
    numeric_series = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = numeric_series.quantile(0.75)  # filter above 75th percentile
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f} (top quartile): {len(filtered_df)} records")
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (numeric_series[numeric_series > threshold] - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field if present
    group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between numeric fields in your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and numeric_field:
    df = dataframes[main_record_set_id]
    numeric_series = pd.to_numeric(df[numeric_field], errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(numeric_series, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # If a group_field was found:
    if 'group_field' in locals():
        plt.figure(figsize=(9,4))
        sns.boxplot(x=df[group_field], y=numeric_series)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR^2 dataset 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' using `mlcroissant`. We examined the available record sets and fields (referenced by their `@id` values), extracted their tabular content, performed basic exploratory data analyses including filtering and normalization, and visualized the distributions of selected fields.

This approach enables FAIR, reproducible data exploration on Croissant-compliant datasets. For deeper insights, consult the full metadata, experiment with additional fields, or try advanced machine learning methods!
